In [ ]:
MODEL_NAME = "csgo-match-predictor"
NAMESPACE  = "default"
YAML_FILE  = "inference_service.yaml"

In [ ]:
import subprocess

def sh(cmd: str) -> str:
    print(f"$ {cmd}")
    result = subprocess.run(cmd, shell=True, check=True, capture_output=True, text=True)
    if result.stdout:
        print(result.stdout)
    return result.stdout

sh(f"kubectl apply -f {YAML_FILE} -n {NAMESPACE}")
sh(f"kubectl get inferenceservice {MODEL_NAME} -n {NAMESPACE}")

In [ ]:
sh(
    f"kubectl wait inferenceservice/{MODEL_NAME} "
    f"--for=condition=Ready -n {NAMESPACE} --timeout=300s"
)
sh(f"kubectl get inferenceservice {MODEL_NAME} -n {NAMESPACE}")

In [ ]:
import json

raw = sh(f"kubectl get inferenceservice {MODEL_NAME} -n {NAMESPACE} -o json")
svc = json.loads(raw)
url = svc["status"]["url"]

PREDICT_URL = f"{url}/v1/models/{MODEL_NAME}:predict"
print(f"Endpoint: {PREDICT_URL}")
print(f"Set INFERENCE_URL={PREDICT_URL} in 07's .env")